# Baseline Experiments: WN18RR & YAGO3-10

**Run overnight on Google Colab with GPU runtime.**

This notebook runs MC Dropout and Deep Ensemble baselines on:
- WN18RR (~30 min)
- YAGO3-10 (~2 hrs)

Expected total runtime: ~2.5-3 hours

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import roc_auc_score
import json
import os
import random
import urllib.request
from datetime import datetime

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Started at: {datetime.now()}")

In [ ]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 2048,
    'lr': 0.001,
    'dropout': 0.3,
    'mc_samples': 20,
    'n_ensembles': 5,
    'seeds': [42, 123, 456],
}

# CAGP reference results from our experiments
CAGP_RESULTS = {
    'WN18RR': {'mean': 0.871, 'std': 0.003},
    'YAGO3-10': {'mean': 0.942, 'std': 0.000},
}

## Data Loading

In [ ]:
def download_wn18rr():
    """Download WN18RR dataset."""
    os.makedirs('data/wn18rr', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/wn18rr"
    
    for split in ['train', 'test']:
        path = f'data/wn18rr/{split}.txt'
        if not os.path.exists(path):
            print(f"Downloading WN18RR {split}...")
            urllib.request.urlretrieve(f"{base_url}/{split}.txt", path)
    print("WN18RR download complete!")


def download_yago310():
    """Download YAGO3-10 dataset (OpenKE format)."""
    os.makedirs('data/yago310', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/thunlp/OpenKE/OpenKE-PyTorch/benchmarks/YAGO3-10"
    
    for f in ['train2id.txt', 'test2id.txt', 'entity2id.txt', 'relation2id.txt']:
        path = f'data/yago310/{f}'
        if not os.path.exists(path):
            print(f"Downloading YAGO3-10 {f}...")
            urllib.request.urlretrieve(f"{base_url}/{f}", path)
    print("YAGO3-10 download complete!")


def load_triples_standard(path):
    """Load triples from standard format (h\tr\tt per line)."""
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples


def load_triples_openke(path):
    """Load triples from OpenKE format (first line is count, then h t r per line)."""
    triples = []
    with open(path) as f:
        n = int(f.readline().strip())
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                h, t, r = parts[0], parts[1], parts[2]  # OpenKE format: h t r
                triples.append((h, r, t))  # Convert to (h, r, t)
    return triples


def load_dataset(name):
    """Load a dataset by name."""
    if name == 'WN18RR':
        download_wn18rr()
        train = load_triples_standard('data/wn18rr/train.txt')
        test = load_triples_standard('data/wn18rr/test.txt')
    elif name == 'YAGO3-10':
        download_yago310()
        train = load_triples_openke('data/yago310/train2id.txt')
        test = load_triples_openke('data/yago310/test2id.txt')
    else:
        raise ValueError(f"Unknown dataset: {name}")
    
    # Build entity and relation mappings
    entities = set()
    relations = set()
    for h, r, t in train + test:
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    ent2idx = {e: i for i, e in enumerate(entities)}
    rel2idx = {r: i for i, r in enumerate(relations)}
    
    print(f"{name}: {len(train)} train, {len(test)} test")
    print(f"Entities: {len(entities)}, Relations: {len(relations)}")
    
    return train, test, ent2idx, rel2idx

## Model Definitions

In [ ]:
class DistMultDropout(nn.Module):
    """DistMult with dropout for MC Dropout uncertainty."""
    def __init__(self, num_entities, num_relations, dim, dropout=0.3):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.dropout(self.entity_emb(heads))
        r = self.relation_emb(relations)
        t = self.dropout(self.entity_emb(tails))
        return (h * r * t).sum(dim=-1)

    def get_uncertainty_mc(self, heads, relations, tails, n_samples=20):
        """MC Dropout uncertainty: predictive entropy."""
        self.train()  # Enable dropout
        scores = []
        with torch.no_grad():
            for _ in range(n_samples):
                score = torch.sigmoid(self.forward(heads, relations, tails))
                scores.append(score)
        scores = torch.stack(scores)
        mean_score = scores.mean(dim=0)
        entropy = -mean_score * torch.log(mean_score + 1e-10) - (1-mean_score) * torch.log(1-mean_score + 1e-10)
        return entropy


class DistMult(nn.Module):
    """Standard DistMult for ensemble."""
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)


class DeepEnsemble:
    """Ensemble of DistMult models."""
    def __init__(self, models):
        self.models = models

    def get_uncertainty(self, heads, relations, tails):
        """Ensemble uncertainty: predictive entropy."""
        scores = []
        for model in self.models:
            model.eval()
            with torch.no_grad():
                score = torch.sigmoid(model(heads, relations, tails))
                scores.append(score)
        scores = torch.stack(scores)
        mean_score = scores.mean(dim=0)
        entropy = -mean_score * torch.log(mean_score + 1e-10) - (1-mean_score) * torch.log(1-mean_score + 1e-10)
        return entropy

## Training & Evaluation

In [ ]:
def train_model(model, triples, ent2idx, rel2idx, epochs, verbose=True):
    """Train a model on the given triples."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    heads = torch.tensor([ent2idx[h] for h, r, t in triples])
    relations = torch.tensor([rel2idx[r] for h, r, t in triples])
    tails = torch.tensor([ent2idx[t] for h, r, t in triples])

    loader = DataLoader(
        TensorDataset(heads, relations, tails),
        batch_size=CONFIG['batch_size'], shuffle=True
    )

    model.train()
    for epoch in range(epochs):
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos = model(batch_h, batch_r, batch_t)
            neg_t = torch.randint(0, len(ent2idx), batch_t.shape, device=device)
            neg = model(batch_h, batch_r, neg_t)

            loss = criterion(pos, torch.ones_like(pos)) + criterion(neg, torch.zeros_like(neg))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if verbose and (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{epochs}")

    return model


def evaluate_auroc(get_uncertainty_fn, test, ent2idx, rel2idx):
    """Evaluate AUROC for OOD detection."""
    # Handle both string and int keys
    if isinstance(list(ent2idx.keys())[0], str):
        heads = torch.tensor([ent2idx.get(h, 0) for h, r, t in test]).to(device)
        relations = torch.tensor([rel2idx.get(r, 0) for h, r, t in test]).to(device)
        tails = torch.tensor([ent2idx.get(t, 0) for h, r, t in test]).to(device)
    else:
        heads = torch.tensor([ent2idx.get(int(h), 0) for h, r, t in test]).to(device)
        relations = torch.tensor([rel2idx.get(int(r), 0) for h, r, t in test]).to(device)
        tails = torch.tensor([ent2idx.get(int(t), 0) for h, r, t in test]).to(device)

    with torch.no_grad():
        id_unc = get_uncertainty_fn(heads, relations, tails).cpu().numpy()
        neg_tails = torch.randint(0, len(ent2idx), tails.shape, device=device)
        ood_unc = get_uncertainty_fn(heads, relations, neg_tails).cpu().numpy()

    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])  # Lower uncertainty = more likely ID
    return roc_auc_score(labels, scores)

## Run All Experiments

In [ ]:
all_results = {}

for dataset_name in ['WN18RR', 'YAGO3-10']:
    print(f"\n{'='*70}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*70}")
    print(f"Started at: {datetime.now()}")
    
    # Load dataset
    train, test, ent2idx, rel2idx = load_dataset(dataset_name)
    
    results = {
        'MCDropout': [],
        'DeepEnsemble': [],
    }
    
    for seed in CONFIG['seeds']:
        print(f"\n--- Seed {seed} ---")
        
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)
        
        # MC Dropout
        print("\n  MC Dropout...")
        mc_model = DistMultDropout(
            len(ent2idx), len(rel2idx), 
            CONFIG['embedding_dim'], CONFIG['dropout']
        )
        mc_model = train_model(mc_model, train, ent2idx, rel2idx, CONFIG['epochs'])
        auroc = evaluate_auroc(
            lambda h, r, t: mc_model.get_uncertainty_mc(h, r, t, CONFIG['mc_samples']),
            test, ent2idx, rel2idx
        )
        results['MCDropout'].append(auroc)
        print(f"    AUROC: {auroc:.4f}")
        
        # Deep Ensemble
        print("\n  Deep Ensemble...")
        ensemble_models = []
        for i in range(CONFIG['n_ensembles']):
            torch.manual_seed(seed + i * 1000)
            model = DistMult(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
            model = train_model(model, train, ent2idx, rel2idx, CONFIG['epochs'], verbose=False)
            ensemble_models.append(model)
            print(f"    Trained ensemble model {i+1}/{CONFIG['n_ensembles']}")
        ensemble = DeepEnsemble(ensemble_models)
        auroc = evaluate_auroc(ensemble.get_uncertainty, test, ent2idx, rel2idx)
        results['DeepEnsemble'].append(auroc)
        print(f"    AUROC: {auroc:.4f}")
        
        # Clear GPU memory
        del mc_model, ensemble_models, ensemble
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    all_results[dataset_name] = results
    
    # Print summary for this dataset
    print(f"\n{'-'*50}")
    print(f"{dataset_name} SUMMARY")
    print(f"{'-'*50}")
    for method in results:
        mean = np.mean(results[method])
        std = np.std(results[method])
        print(f"  {method:<15} {mean:.3f} ± {std:.3f}")
    print(f"  {'CAGP (ref)':<15} {CAGP_RESULTS[dataset_name]['mean']:.3f}")
    print(f"Finished at: {datetime.now()}")

## Final Summary

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS FOR PAPER")
print("="*70)
print(f"\n{'Method':<15} {'WN18RR':<12} {'YAGO3-10':<12}")
print("-"*40)

for method in ['MCDropout', 'DeepEnsemble']:
    wn_mean = np.mean(all_results['WN18RR'][method])
    yago_mean = np.mean(all_results['YAGO3-10'][method])
    print(f"{method:<15} {wn_mean:.3f}        {yago_mean:.3f}")

print("-"*40)
print(f"{'CAGP (ours)':<15} {CAGP_RESULTS['WN18RR']['mean']:.3f}        {CAGP_RESULTS['YAGO3-10']['mean']:.3f}")

print("\n" + "="*70)
print("LATEX TABLE ROW FORMAT")
print("="*70)
for method in ['MCDropout', 'DeepEnsemble']:
    wn = np.mean(all_results['WN18RR'][method])
    yago = np.mean(all_results['YAGO3-10'][method])
    # FB15k-237 values from previous run
    fb = 0.430 if method == 'MCDropout' else 0.225
    name = 'MC Dropout' if method == 'MCDropout' else 'Deep Ensemble'
    print(f"{name} & {wn:.3f} & {fb:.3f} & {yago:.3f} \\\\")

In [ ]:
# Save all results
output = {
    'timestamp': str(datetime.now()),
    'config': CONFIG,
    'results': {
        dataset: {
            method: {
                'mean': float(np.mean(all_results[dataset][method])),
                'std': float(np.std(all_results[dataset][method])),
                'values': [float(v) for v in all_results[dataset][method]]
            }
            for method in all_results[dataset]
        }
        for dataset in all_results
    },
    'cagp_reference': CAGP_RESULTS
}

with open('baseline_results_all.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\nResults saved to baseline_results_all.json")
print(json.dumps(output, indent=2))

In [ ]:
# Push to GitHub (optional - uncomment if you want auto-push)
# !git config --global user.email "choroklee@kaist.ac.kr"
# !git config --global user.name "Chorok Lee"
# !git add baseline_results_all.json
# !git commit -m "Add baseline results for WN18RR and YAGO3-10"
# !git push